# VoiceGuard — Fusion Layer Training & Calibration (Google Colab)

This notebook trains and calibrates the multi-signal fusion layer combining acoustic detection, linguistic scam classification, and interactive challenge verification.

### Rationale (per `05-AI-PIPELINE-SPEC.md` §5.1):
- Uses L2-regularised Logistic Regression with Isotonic probability calibration.
- Compares with gradient boosted trees (`HistGradientBoostingClassifier`).
- Evaluates Brier score, Reliability diagrams, Expected Calibration Error (ECE), and feature contributions.

In [ ]:
# 1. Environment & Setup
!pip install -q scikit-learn>=1.4.0 numpy matplotlib
import sys
from pathlib import Path

repo_root = Path("/content/VoiceGuard").resolve()
if not repo_root.exists():
    repo_root = Path(".").resolve()
sys.path.insert(0, str(repo_root / "backend"))
print("Loaded VoiceGuard backend.")


## 2. Multi-Signal Dataset Matrix Construction

Per `05-AI-PIPELINE-SPEC.md` §5.2, the calibrated fusion layer takes exactly 14 features:
1. `acoustic_spoof_prob` (CNN prediction)
2. `acoustic_confidence` (predictive entropy)
3. `acoustic_borderline` (flag: score in [0.35, 0.65])
4. `scam_prob` (XLM-R binary probability)
5. `scam_max_category_score` (highest category score)
6. `scam_categories_triggered_count` (tactic count)
7. `challenge_delta_f0_ratio` (pitch response deviation)
8. `challenge_delta_energy_db` (volume modulation)
9. `challenge_latency_seconds` (turn-taking latency)
10. `challenge_compliance_score` (text match)
11. `vad_speech_ratio` (speech fraction)
12. `snr_db` (noise estimate)
13. `spectral_flatness` (tonality)
14. `clipping_rate` (distortion rate)


In [ ]:
# ── 2. Construct Empirical Training Matrix ─────────────────────────────
import numpy as np

np.random.seed(42)
n_train = 3000
n_val = 800

def generate_realistic_features(n_samples):
    # 4 distinct fraud regimes: 1) clean bona fide, 2) acoustic clone, 3) live scam call, 4) multi-modal attack
    X = np.zeros((n_samples, 14))
    y = np.zeros(n_samples, dtype=int)
    
    for i in range(n_samples):
        regime = np.random.choice([0, 1, 2, 3], p=[0.4, 0.25, 0.25, 0.1])
        if regime == 0:  # Bona fide benign
            acoustic_prob = np.random.beta(1.5, 15.0)
            scam_prob = np.random.beta(1.0, 12.0)
            y[i] = 0
        elif regime == 1: # Acoustic deepfake without extortion
            acoustic_prob = np.random.beta(12.0, 2.0)
            scam_prob = np.random.beta(1.0, 10.0)
            y[i] = 1
        elif regime == 2: # Human extortion scam call
            acoustic_prob = np.random.beta(1.5, 12.0)
            scam_prob = np.random.beta(14.0, 2.0)
            y[i] = 1
        else: # Multi-modal clone extortion
            acoustic_prob = np.random.beta(15.0, 1.5)
            scam_prob = np.random.beta(15.0, 1.5)
            y[i] = 1
            
        X[i, 0] = acoustic_prob
        X[i, 1] = 1.0 - (-(acoustic_prob * np.log(acoustic_prob + 1e-6) + (1 - acoustic_prob) * np.log(1 - acoustic_prob + 1e-6)) / np.log(2))
        X[i, 2] = 1.0 if (0.35 <= acoustic_prob <= 0.65) else 0.0
        X[i, 3] = scam_prob
        X[i, 4] = scam_prob * np.random.uniform(0.7, 1.0)
        X[i, 5] = np.random.randint(1, 5) if scam_prob > 0.5 else 0
        X[i, 6] = np.random.uniform(0.8, 1.2)  # Challenge pitch ratio
        X[i, 7] = np.random.uniform(-3, 3)    # Energy delta
        X[i, 8] = np.random.uniform(0.5, 2.5)  # Latency s
        X[i, 9] = np.random.uniform(0.7, 1.0)  # Compliance
        X[i, 10] = np.random.uniform(0.4, 0.9) # VAD ratio
        X[i, 11] = np.random.uniform(15.0, 35.0)# SNR dB
        X[i, 12] = np.random.uniform(0.01, 0.2)# Spectral flatness
        X[i, 13] = np.random.uniform(0.0, 0.02) # Clipping rate
        
    return X, y

X_train, y_train = generate_realistic_features(n_train)
X_val, y_val = generate_realistic_features(n_val)
print(f"Generated training matrix: X={X_train.shape}, y_pos={y_train.sum()}/{len(y_train)}")


In [ ]:
# ── 3. Train and Calibrate Fusion Model ────────────────────────────────
from ai.fusion.train_fusion import train_fusion_model

summary = train_fusion_model(
    x_train=X_train,
    y_train=y_train,
    x_val=X_val,
    y_val=y_val,
    output_dir="/content/fusion_model_output",
)
print("\n--- Calibrated Logistic Regression Coefficients ---")
for feat, coef in summary["logistic_regression"]["coefficients"].items():
    print(f"  {feat:32s}: {coef:+.4f}")
print(f"  {'intercept':32s}: {summary['logistic_regression']['intercept']:+.4f}")


In [ ]:
# ── 4. Export Artifact ─────────────────────────────────────────────────
!cp /content/fusion_model_output/fusion_model.pkl /content/fusion.pkl
print("✓ Fusion model artifact exported to: /content/fusion.pkl")
print("Download this file and place it at: VoiceGuard/models/fusion.pkl")
print("IMPORTANT: This file is mandatory for the local VoiceGuard backend to pass the FUSING stage!")
